In [10]:
import math
import matplotlib.pyplot as plt
import random


# Distance cache
_distance_cache = {}


def euclidean_distance(p1, p2):
    key = (p1, p2) if p1 <= p2 else (p2, p1)
    if key not in _distance_cache:
        _distance_cache[key] = math.hypot(p1[0] - p2[0], p1[1] - p2[1])
    return _distance_cache[key]


def read_tsp_data(filename):
    nodes = {}
    with open(filename, 'r') as file:
        lines = file.readlines()
        start_parsing = False
        for line in lines:
            if line.startswith("NODE_COORD_SECTION"):
                start_parsing = True
                continue
            if line.startswith("EOF"):
                break
            if start_parsing:
                parts = line.strip().split()
                nodes[int(parts[0])] = (float(parts[1]), float(parts[2]))
    return nodes




def calculate_tour_length(tour, nodes):
    distance = sum(euclidean_distance(nodes[tour[i]], nodes[tour[i + 1]]) for i in range(len(tour) - 1))
    distance += euclidean_distance(nodes[tour[-1]], nodes[tour[0]])  # Return to start
    return distance

In [11]:
def initialize_pheromone_matrix(nodes):
    n = len(nodes)
    return [[1.0 for _ in range(n)] for _ in range(n)]

def heuristic_value(p1, p2):
    epsilon = 1e-10
    return 1.0 / (euclidean_distance(p1, p2) + epsilon)

def select_next_node(current_node, unvisited, pheromone, nodes, alpha, beta):
    total = 0.0
    probs = []
    for node in unvisited:
        tau = pheromone[current_node][node] ** alpha
        eta = heuristic_value(nodes[current_node + 1], nodes[node + 1]) ** beta
        prob = tau * eta
        probs.append((node, prob))
        total += prob

    if total == 0:
        return random.choice(unvisited)

    r = random.uniform(0, total)
    cumulative = 0
    for node, prob in probs:
        cumulative += prob
        if cumulative >= r:
            return node

    return unvisited[-1]

def construct_solution(pheromone, nodes, num_ants, alpha, beta):
    n = len(nodes)
    tours = []
    for _ in range(num_ants):
        start = random.randint(0, n - 1)
        tour = [start]
        unvisited = list(range(n))
        unvisited.remove(start)

        while unvisited:
            current = tour[-1]
            next_node = select_next_node(current, unvisited, pheromone, nodes, alpha, beta)
            tour.append(next_node)
            unvisited.remove(next_node)

        tours.append(tour)
    return tours

def update_pheromones(pheromone, tours, nodes, evaporation, Q):
    n = len(pheromone)
    # Evaporation
    for i in range(n):
        for j in range(n):
            pheromone[i][j] *= (1 - evaporation)

    # Deposit
    for tour in tours:
        length = calculate_tour_length([t + 1 for t in tour], nodes)
        for i in range(len(tour)):
            a, b = tour[i], tour[(i + 1) % len(tour)]
            delta = Q / length
            pheromone[a][b] += delta
            pheromone[b][a] += delta


In [12]:
def ant_colony_optimization(
    nodes, num_ants=10, num_iterations=10,
    alpha=1.0, beta=5.0, evaporation=0.5, Q=100
):
    n = len(nodes)
    pheromone = initialize_pheromone_matrix(nodes)
    best_tour = None
    best_length = float('inf')

    for iteration in range(num_iterations):
        tours = construct_solution(pheromone, nodes, num_ants, alpha, beta)
        for tour in tours:
            length = calculate_tour_length([t + 1 for t in tour], nodes)
            if length < best_length:
                best_length = length
                best_tour = tour
        update_pheromones(pheromone, tours, nodes, evaporation, Q)

        print(f"Iteration {iteration+1}, Best length so far: {best_length:.2f}")

    return [t + 1 for t in best_tour], best_length


In [13]:
city_nodes = read_tsp_data("bier127.tsp")

# First run of the ant colony optimization algorithm
best_path, best_path_length = ant_colony_optimization(
    city_nodes,
    num_ants=10,
    num_iterations=10,
    alpha=1.0,
    beta=5.0,
    evaporation=0.5,
    Q=100
)

print("\nBest path found (run 1):", best_path)
print("Best path length (run 1):", best_path_length)

# Second run of the ant colony optimization algorithm (with different parameters)
best_path, best_path_length = ant_colony_optimization(
    city_nodes,
    num_ants=10,
    num_iterations=10,
    alpha=1.5,
    beta=4.0,
    evaporation=0.7,
    Q=100
)

print("\nBest path found (run 2):", best_path)
print("Best path length (run 2):", best_path_length)


Iteration 1, Best length so far: 156284.87
Iteration 2, Best length so far: 156284.87
Iteration 3, Best length so far: 156284.87
Iteration 4, Best length so far: 156284.87
Iteration 5, Best length so far: 156284.87
Iteration 6, Best length so far: 156284.87
Iteration 7, Best length so far: 156284.87
Iteration 8, Best length so far: 156284.87
Iteration 9, Best length so far: 156284.87
Iteration 10, Best length so far: 156284.87

Best path found (run 1): [4, 24, 23, 19, 22, 20, 108, 17, 21, 18, 77, 74, 73, 68, 71, 70, 69, 75, 78, 79, 80, 31, 27, 30, 41, 14, 12, 35, 37, 36, 43, 40, 34, 39, 38, 26, 25, 33, 122, 28, 29, 32, 8, 72, 114, 11, 9, 6, 106, 15, 7, 105, 120, 50, 2, 51, 57, 54, 45, 103, 44, 48, 118, 53, 49, 47, 46, 124, 52, 56, 5, 121, 115, 13, 10, 3, 58, 64, 100, 61, 62, 60, 116, 90, 91, 59, 67, 76, 117, 84, 81, 83, 82, 126, 96, 109, 87, 88, 86, 85, 110, 104, 125, 89, 92, 99, 65, 113, 66, 55, 1, 16, 42, 123, 95, 97, 127, 93, 94, 111, 112, 107, 98, 101, 102, 63, 119]
Best path lengt

### Ant Colony System (ACS) - Changes to Edge Selection

In the **Ant Colony System (ACS)**, three significant changes are made to the original **Ant System (AS)**:

1. **Edge Selection is Biased Towards Exploitation**: The ACS algorithm introduces a biased edge selection where the probability of selecting the next node is skewed toward exploiting the best path (i.e., the shortest edges with higher pheromone levels).
2. **Local Pheromone Updates**: While constructing a solution, ants immediately update the pheromone level of the edges they traverse using a **local pheromone update** rule. This encourages exploration of different paths within the same iteration.
3. **Global Pheromone Update**: After each iteration, only the best solution found in that iteration will contribute to updating the global pheromone trails. This prevents less successful solutions from affecting the pheromone trails.

### Changes in `select_next_node()` Function:

The `select_next_node()` function is modified to incorporate the key features of ACS:

- **Greedy Exploitation (`q₀`)**: If a random number is less than `q₀`, the algorithm **greedily selects** the next node that maximizes the pheromone level and heuristic value. This biases the selection towards the best available path.
- **Probabilistic Exploration**: When the random number exceeds `q₀`, the ant chooses the next node probabilistically, similar to the original ACO.
- **Local Pheromone Update**: After selecting the next node, the pheromone on the edge is updated immediately, reflecting the local pheromone modification rule that encourages diversity in paths.



In [14]:
def select_next_node(current_node, unvisited, pheromone, nodes, alpha, beta, q0, tau0):
    q = random.random()
    if q < q0:
        # Greedy exploitation
        next_node = max(
            unvisited,
            key=lambda node: pheromone[current_node][node] * (heuristic_value(nodes[current_node + 1], nodes[node + 1]) ** beta)
        )
    else:
        # Probabilistic exploration
        total = 0.0
        probs = []
        for node in unvisited:
            tau = pheromone[current_node][node] ** alpha
            eta = heuristic_value(nodes[current_node + 1], nodes[node + 1]) ** beta
            prob = tau * eta
            probs.append((node, prob))
            total += prob

        if total == 0:
            return random.choice(unvisited)

        r = random.uniform(0, total)
        cumulative = 0
        for node, prob in probs:
            cumulative += prob
            if cumulative >= r:
                next_node = node
                break
        else:
            next_node = unvisited[-1]

    # Local pheromone update
    pheromone[current_node][next_node] = (1 - 0.1) * pheromone[current_node][next_node] + 0.1 * tau0
    pheromone[next_node][current_node] = pheromone[current_node][next_node]  # Symmetric

    return next_node


### Changes in `construct_solution()` Function:

The `construct_solution()` function is responsible for constructing a solution (tour) for each ant. In the Ant Colony System (ACS), the following key changes are applied:

- **Local pheromone updates**: During the construction of the solution, ants immediately update the pheromone levels on the edges they traverse, as discussed previously in the `select_next_node()` function.
- **Greedy and Probabilistic Selection**: The ants select the next node based on the **greedy exploitation** rule (`q₀` probability) or by **probabilistic exploration** when `random() > q₀`.

Here is the updated `construct_solution()` function for ACS:
### Changes in `construct_solution()` Function:

The `construct_solution()` function is responsible for constructing a solution (tour) for each ant. In the Ant Colony System (ACS), the following key changes are applied:

- **Local pheromone updates**: During the construction of the solution, ants immediately update the pheromone levels on the edges they traverse, as discussed previously in the `select_next_node()` function.
- **Greedy and Probabilistic Selection**: The ants select the next node based on the **greedy exploitation** rule (`q₀` probability) or by **probabilistic exploration** when `random() > q₀`.

Here is the updated `construct_solution()` function for ACS:


In [15]:
def construct_solution(pheromone, nodes, num_ants, alpha, beta, q0, tau0):
    n = len(nodes)
    tours = []
    for _ in range(num_ants):
        start = random.randint(0, n - 1)
        tour = [start]
        unvisited = list(range(n))
        unvisited.remove(start)

        while unvisited:
            current = tour[-1]
            next_node = select_next_node(current, unvisited, pheromone, nodes, alpha, beta, q0, tau0)
            tour.append(next_node)
            unvisited.remove(next_node)

        tours.append(tour)
    return tours


### Changes in `update_pheromones()` Function:

The `update_pheromones()` function is responsible for updating the pheromone matrix after each iteration. In ACS, this update occurs in two steps:

1. **Evaporation**: The pheromone levels evaporate for all edges to simulate pheromone decay over time. This ensures that less frequently used paths gradually lose their attractiveness.
2. **Global pheromone update**: Only the best tour found in the current iteration contributes to the global pheromone update. The pheromone on the edges of the best tour is increased, reinforcing the successful path and encouraging other ants to follow it in future iterations.

This function encapsulates the key idea that **only the best solution from the current iteration** is used to update the pheromone trail globally, which is a hallmark of the ACS approach.


In [16]:
def update_pheromones(pheromone, best_tour, nodes, evaporation, Q):
    n = len(pheromone)
    # Evaporation
    for i in range(n):
        for j in range(n):
            pheromone[i][j] *= (1 - evaporation)

    # Global update by best tour only
    length = calculate_tour_length([t + 1 for t in best_tour], nodes)
    for i in range(len(best_tour)):
        a, b = best_tour[i], best_tour[(i + 1) % len(best_tour)]
        delta = Q / length
        pheromone[a][b] += delta
        pheromone[b][a] += delta


### Changes in `ant_colony_system()` Function:

The `ant_colony_system()` function implements the full ACS algorithm. It coordinates the construction of solutions by ants, pheromone updates, and the selection of the best tour found. Here’s a breakdown of the modifications and functionality:

1. **Solution Construction**: Ants construct solutions using the `construct_solution()` function, which combines **greedy exploitation** and **probabilistic exploration** based on the `q₀` parameter.
2. **Pheromone Updates**: The pheromone matrix is updated in two stages: **local pheromone updates** during solution construction and **global pheromone updates** at the end of the iteration, using only the best solution found in that iteration.
3. **Best Solution Tracking**: The function tracks the best tour found across all iterations and updates the pheromone matrix accordingly.
4. **Termination**: The algorithm terminates after a fixed number of iterations, returning the best tour and its length.

Key parameters:
- `q₀`: Exploitation probability for greedy selection (default is 0.9).
- `tau₀`: Initial pheromone value (default is 1.0).
- `num_ants`, `num_iterations`, `alpha`, `beta`, `evaporation`, and `Q` control various aspects of the search process, such as the number of ants, pheromone decay, and the relative influence of pheromone vs. heuristic information.


In [17]:
def ant_colony_system(
    nodes, num_ants=10, num_iterations=10,
    alpha=1.0, beta=5.0, evaporation=0.5, Q=100,
    q0=0.9, tau0=1.0
):
    n = len(nodes)
    pheromone = initialize_pheromone_matrix(nodes)
    best_tour = None
    best_length = float('inf')

    best_over_iter = []
    for iteration in range(num_iterations):
        tours = construct_solution(pheromone, nodes, num_ants, alpha, beta, q0, tau0)
        for tour in tours:
            length = calculate_tour_length([t + 1 for t in tour], nodes)
            if length < best_length:
                best_length = length
                best_tour = tour
        update_pheromones(pheromone, best_tour, nodes, evaporation, Q)
        best_over_iter.append(best_length)
        print(f"Iteration {iteration+1}, Best length so far: {best_length:.2f}")

    return [t + 1 for t in best_tour], best_length, best_over_iter


## ACS Parameter Testing and Convergence Plotting

In this section, we evaluate the behavior of the Ant Colony System (ACS) algorithm using different parameter configurations on two Traveling Salesman Problem (TSP) datasets: `bier127.tsp` and `nu3496.tsp`.

### Experimental Setup

We define a list of parameter sets for ACS:
- **num_ants**: Number of ants used per iteration
- **num_iterations**: Number of iterations to run the algorithm
- **alpha (α)**: Relative importance of the pheromone trail
- **beta (β)**: Relative importance of the heuristic information (e.g., inverse distance)
- **evaporation**: Global pheromone evaporation rate
- **Q**: Constant used in the global pheromone update rule
- **q0**: Probability of exploitation vs. exploration
- **tau0**: Initial pheromone value

For each parameter configuration, we run the algorithm and collect the best tour length found after each iteration. These lengths are then plotted using Plotly to visualize the convergence behavior.

### Results

- The **first plot** shows convergence curves for different parameter sets on `bier127.tsp`, a moderate-sized TSP instance.
- The **second plot** shows convergence on the larger `nu3496.tsp` dataset with smaller-scale parameter settings for faster runtime.


In [18]:
import plotly.graph_objects as go

# Define parameter sets: (num_ants, num_iterations, alpha, beta, evaporation, Q, q0, tau0)
parameter_sets = [
    (10, 50, 1.0, 5.0, 0.5, 100, 0.9, 1.0),
    (20, 50, 1.5, 4.0, 0.6, 120, 0.8, 1.2),
    (30, 50, 2.0, 3.0, 0.7, 150, 0.7, 1.5)
]

# Load TSP datasets using your read_tsp_data function
tsp_nodes_bier = read_tsp_data("bier127.tsp")
tsp_nodes_large = read_tsp_data("nu3496.tsp")

# Run ACS for each parameter set and collect length progression
def run_tests_and_plot(nodes, dataset_name):
    fig = go.Figure()

    for params in parameter_sets:
        num_ants, num_iterations, alpha, beta, evaporation, Q, q0, tau0 = params
        best_tour, best_length, best_over_iter = ant_colony_system(
            nodes,
            num_ants=num_ants,
            num_iterations=num_iterations,
            alpha=alpha,
            beta=beta,
            evaporation=evaporation,
            Q=Q,
            q0=q0,
            tau0=tau0
        )

        fig.add_trace(go.Scatter(
            y=best_over_iter,
            mode='lines',
            name=f"ants={num_ants}, α={alpha}, β={beta}, q₀={q0}"
        ))

    fig.update_layout(
        title=f"ACS Best Tour Lengths over Iterations ({dataset_name})",
        xaxis_title="Iteration",
        yaxis_title="Best Tour Length",
        legend_title="Parameter Sets",
        template="plotly_white"
    )
    fig.show()

# Run and plot for both datasets
run_tests_and_plot(tsp_nodes_bier, "bier127.tsp")

parameter_sets = [
    (2, 5, 1.0, 5.0, 0.5, 100, 0.9, 1.0),
    (2, 5, 1.5, 4.0, 0.6, 120, 0.8, 1.2),
    (2, 5, 2.0, 3.0, 0.7, 150, 0.7, 1.5)
]


run_tests_and_plot(tsp_nodes_large, "nu3496.tsp")


Iteration 1, Best length so far: 135567.78
Iteration 2, Best length so far: 129075.20
Iteration 3, Best length so far: 129075.20
Iteration 4, Best length so far: 129075.20
Iteration 5, Best length so far: 129075.20
Iteration 6, Best length so far: 129075.20
Iteration 7, Best length so far: 129075.20
Iteration 8, Best length so far: 129075.20
Iteration 9, Best length so far: 129075.20
Iteration 10, Best length so far: 129075.20
Iteration 11, Best length so far: 129075.20
Iteration 12, Best length so far: 129075.20
Iteration 13, Best length so far: 129075.20
Iteration 14, Best length so far: 129075.20
Iteration 15, Best length so far: 129075.20
Iteration 16, Best length so far: 129075.20
Iteration 17, Best length so far: 129075.20
Iteration 18, Best length so far: 129075.20
Iteration 19, Best length so far: 129075.20
Iteration 20, Best length so far: 129075.20
Iteration 21, Best length so far: 129075.20
Iteration 22, Best length so far: 129075.20
Iteration 23, Best length so far: 129075.

Iteration 1, Best length so far: 124155.45
Iteration 2, Best length so far: 123430.78
Iteration 3, Best length so far: 123430.78
Iteration 4, Best length so far: 118590.86
Iteration 5, Best length so far: 118590.86
Iteration 1, Best length so far: 134920.88
Iteration 2, Best length so far: 132397.18
Iteration 3, Best length so far: 132397.18
Iteration 4, Best length so far: 132397.18
Iteration 5, Best length so far: 124029.69
Iteration 1, Best length so far: 169007.78
Iteration 2, Best length so far: 169007.78
Iteration 3, Best length so far: 150554.81
Iteration 4, Best length so far: 138681.19
Iteration 5, Best length so far: 138681.19


## Final Conclusions

The Ant Colony System (ACS) algorithm was evaluated on two datasets: `bier127.tsp` and `nu3496.tsp`.

- On the **bier127.tsp** dataset (127 cities), ACS produced **worse results** compared to other metaheuristics used in previous assignments such as **Tabu Search** and the **Genetic Algorithm**. This underperformance is likely due to the strong **dependence on parameter tuning**. The parameters used here were not thoroughly optimized, and better results might be achievable with more extensive tuning or adaptive techniques.

- For the **nu3496.tsp** dataset (3496 cities), the scalability issues of ACS became evident. Because the algorithm has a time complexity of approximately **O(n²)** per ant per iteration, increasing the number of ants or cities **dramatically impacts execution time**. As a result, for large-scale instances, ACS can become computationally infeasible without significant optimization or parallelization.

In summary, while ACS is a powerful algorithm inspired by nature, its effectiveness is **highly sensitive to parameter choices** and **does not scale well** without careful engineering. It remains a valuable technique for medium-sized problems or when well-tuned, but may be outperformed by more efficient or adaptive methods in large-scale scenarios.
